<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Network Diagnostics and Testing

Use network and diagnostic tools only on computers and networks you own or are authorized to administer. Begin with safe localhost checks, then use small targeted checks for a known Duckiedrone rather than broad scans. A dashboard that will not open or a Duckiedrone that does not reply to `ping` provides little information by itself: the hostname might not resolve, the network path might be unavailable, or the intended service might not be listening.

This notebook uses a short evidence sequence to narrow down a failure, beginning with a connection you control locally and then checking one authorized Duckiedrone service.

## Inspect the current machine

These read-only commands show where the command runs, active interfaces, and the routing table:

```bash
hostname
ip -brief address
ip route
```

[Notebook 17](./17-network-addressing-and-routing.ipynb) introduced `ip -brief address` for interface names and assigned addresses, and `ip route` for the local routing table. In `ip route` output, the line beginning with `default via` identifies the default gateway. A network interface can have IPv4 and IPv6 addresses at once, called dual stack. In the commands below, `-4` selects IPv4 addresses and `-6` selects IPv6 addresses:

```bash
ip -4 -brief address
ip -6 -brief address
```

In a local Linux shell, loopback normally shows `127.0.0.1/8` for IPv4 and `::1/128` for IPv6; both refer to the machine running the command. On a base station, a wired or Wi-Fi interface can show additional address families. An IPv6 address beginning `fe80::` is link-local and cannot cross a router. Do not disable IPv6 or manually change addresses to work around a failure.

## Resolve names and inspect routes

[Notebook 13](./13-physical-duckiedrone-ssh-access.ipynb) introduced `getent hosts` as a lookup for one known name through configured name-resolution sources. The first command is a safe localhost example. Run the second only for an authorized Duckiedrone on the intended local network:

```bash
getent hosts localhost
getent hosts DUCKIEDRONE_NAME.local
```

A hostname lookup is separate from contacting a Duckiedrone; if it fails, verify the intended network and whether DNS or mDNS is allowed.

`ip route get` consults the local routing table without sending a packet. The first command asks how to reach the loopback address, `127.0.0.1`; its result should show the `lo` loopback interface. For the second, replace `KNOWN_DUCKIEDRONE_IP` with the known numeric address of a Duckiedrone you are authorized to inspect:

```bash
ip route get 127.0.0.1
ip route get KNOWN_DUCKIEDRONE_IP
```

For an authorized Duckiedrone, the result identifies the outgoing interface and source address, and it can show `via GATEWAY`. No `via` can be normal on the same subnet. Use ordinary IPv4 or non-link-local IPv6 here; a `fe80::` address needs an interface name and belongs in documented IPv6 guidance.

## Discover compatible local devices

[Notebook 21](./21-network-names-and-service-discovery.ipynb) introduced `dts fleet discover` and the limits of its discovery evidence. From the separate base-station terminal, run this authorized local-device check when discovery evidence is needed:

```bash
dts fleet discover
```

## Test reachability with a local listener

[Notebook 13](./13-physical-duckiedrone-ssh-access.ipynb) introduced bounded ICMP reachability checks with `ping`. The first is safe localhost testing; target `DUCKIEDRONE_NAME.local` only when authorized:

```bash
ping -c 3 127.0.0.1
ping -c 3 DUCKIEDRONE_NAME.local
```

A failed ping does not prove a machine is absent: a firewall can block ICMP and name resolution can fail first. A successful ping does not prove an SSH service or another TCP service is available.

### Try it

Build evidence for the current local context before testing any remote device:

```bash
hostname
ip route get 127.0.0.1
ping -c 3 127.0.0.1
```

Record what each result establishes: the command context, selected loopback route, and local ICMP reachability. Only after those checks, and only with authorization, use the notebook's targeted Duckiedrone checks for one known hostname and one documented service.

<details>
<summary>Check your result</summary>

The three localhost checks do not contact a Duckiedrone. They establish a safe baseline in the current context. A later failure against a known Duckiedrone can then be investigated as a name-resolution, route, reachability, policy, or service problem without broadening the scope of the test.

</details>

[Notebook 19](./19-localhost-and-service-binding.ipynb) introduced `nc`. Here, `-l` makes it listen for one incoming TCP connection. In the following loopback example, the first command listens and the second opens a connection:

To test TCP safely on one machine, open two terminals in the same local context. In the first, start a temporary listener:

```bash
nc -l 127.0.0.1 12345
```

In the second, connect through loopback:

```bash
nc 127.0.0.1 12345
```

Type a line and press `Enter` in either terminal; it appears in the other. Press `Ctrl-C` in both terminals when finished. While the listener runs, a third terminal in the same context can inspect it. [Notebook 18](./18-network-protocols-and-quality.ipynb) introduced `ss -ltn`, and [Notebook 19](./19-localhost-and-service-binding.ipynb) introduced its local-port filter:

```bash
ss -ltn 'sport = :12345'
```

Here, `LISTEN` means a process waits to accept connections. `127.0.0.1:12345` accepts only loopback IPv4, `0.0.0.0:12345` accepts matching IPv4 addresses in its context, and `[::]:12345` has the IPv6 wildcard meaning. A listener alone does not prove another computer can reach it because routing and firewall rules still apply.

## Test one known authorized service

After resolving a hostname and confirming the intended route, test only a documented port that you are authorized to contact. In `nc -vz -w 3`, `-v` reports the result, `-z` performs a service check without sending application data, and `-w 3` limits the wait to three seconds. [Notebook 19](./19-localhost-and-service-binding.ipynb) introduced `curl --head`; here, `--max-time 5` limits the total wait to five seconds:

```bash
nc -vz -w 3 DUCKIEDRONE_NAME.local 22
curl --head --max-time 5 http://DUCKIEDRONE_NAME.local
```

An HTTP response status line beginning with `HTTP/` proves a web server responded, while `2xx`, `3xx`, `401`, `403`, and `404` provide different application evidence. A `405 Method Not Allowed` can still show the dashboard is available. Do not scan hosts or ports, probe unfamiliar paths, or bypass HTTPS certificate verification.

For the TCP check, a result containing `succeeded` means a TCP connection completed to that address and port. `Connection refused` means the attempt received an explicit refusal: the host may be reachable with no service listening, but a firewall or another network device can also reject it. A timeout means the connection did not complete; it does not by itself identify the cause.

## Follow the diagnosis chain

For a dashboard failure, the appropriate evidence order is shown in [Figure 1](#figure-1).

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    correct terminal and network
      |
      v
    name resolves to an address
      |
      v
    route selected
      |
      v
    connection to the intended service
      |
      v
    application response
  </pre>
  <figcaption>Figure 1: Evidence sequence for an authorized dashboard check.</figcaption>
</figure>

When a connection does not work, use the applicable checks in this order:

1. From the base-station terminal, confirm it has an address and default route with `ip`.

2. Confirm the Duckiedrone is powered and connected to the intended network or subnet.

3. If you need to identify a compatible local Duckiedrone, run `dts fleet discover` from the base-station terminal and record its hostname.

4. Resolve the known `DUCKIEDRONE_NAME.local` with `getent hosts`.

5. Inspect the selected route with `ip route get KNOWN_DUCKIEDRONE_IP`.

6. Use authorized `ping` evidence, remembering ICMP can be blocked.

7. Test one known TCP service only with permission.

8. Request headers only from a documented authorized dashboard.

9. Use `ss` to inspect a listening address without changing configuration.

Each result narrows the problem to name resolution, selected route, local discovery, reachability, firewall policy, listening service, or application response.

When requesting help through an available support channel, include the command, its result, where it ran, and the time of the test. Keep credentials, private keys, tokens, and sensitive configuration out of the report.

## Further reading

Manual pages for [getent](https://man7.org/linux/man-pages/man1/getent.1.html), [`ip route`](https://man7.org/linux/man-pages/man8/ip-route.8.html), [`ss`](https://man7.org/linux/man-pages/man8/ss.8.html), [ping](https://man7.org/linux/man-pages/man8/ping.8.html), and curl's [command-line reference](https://curl.se/docs/manpage.html) cover these tools.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
